| Categoria   | Filtro    |
| ----------- | --------- |
| Passa-Baixo | Average   |
| Passa-Baixo | Median    |
| Passa-Baixo | Gaussian  |
| Passa-Alto  | Sobel     |
| Passa-Alto  | Laplacian |
 são estes, estes filtros tem de ser isolados, apenas recebem como parametro a imagem em uint8, e escala de cinzento

# Filtros no domínio espacial

> Cada filtro é **independente** e recebe **apenas** a imagem de entrada (`uint8`, escala de cinzentos).

Implementação pedagógica: convolução manual e janelas explícitas (sem `cv2`).

In [ ]:
%matplotlib inline

# ==========================================================
# BIBLIOTECAS
# ==========================================================

import numpy as np

## Convolução 2D manual

Base comum aos filtros linearmente separáveis (média, Gaussiano, Sobel, Laplaciano).

- entrada com **zero padding**;
- saída intermédia em `float` antes de voltar a `uint8`.

In [ ]:
# ==========================================================
# CONVOLUÇÃO MANUAL (GRAYSCALE)
# ==========================================================

def convolucao_manual(imagem_uint8, kernel):
    """
    Convolui imagem uint8 2D com kernel 2D (float).
    """

    # Validar entrada (contrato do notebook)
    if imagem_uint8.ndim != 2:
        raise ValueError("A imagem deve ser 2D (escala de cinzentos).")
    if imagem_uint8.dtype != np.uint8:
        raise ValueError("A imagem deve estar em uint8.")

    kernel = np.asarray(kernel, dtype=np.float64)
    altura_kernel, largura_kernel = kernel.shape
    pad_y = altura_kernel // 2
    pad_x = largura_kernel // 2

    # Zero padding
    imagem_float = imagem_uint8.astype(np.float64)
    imagem_padded = np.pad(imagem_float, ((pad_y, pad_y), (pad_x, pad_x)), mode="constant")

    altura, largura = imagem_uint8.shape
    resultado = np.zeros((altura, largura), dtype=np.float64)

    for i in range(altura):
        for j in range(largura):
            soma_ponderada = 0.0
            for ky in range(altura_kernel):
                for kx in range(largura_kernel):
                    valor_pixel = imagem_padded[i + ky, j + kx]
                    peso_kernel = kernel[ky, kx]
                    soma_ponderada += valor_pixel * peso_kernel
            resultado[i, j] = soma_ponderada

    return resultado


def limitar_para_uint8(mapa_float):
    """Limita valores float para uint8 [0, 255]."""
    mapa_arredondado = np.round(mapa_float)
    mapa_clip = np.clip(mapa_arredondado, 0, 255)
    return mapa_clip.astype(np.uint8)

### Passa-baixo — Average (média)

Kernel de média $1/(N \times N)$ com tamanho ímpar (ex.: 3×3).

In [ ]:
# ==========================================================
# FILTRO AVERAGE (PASSA-BAIXO)
# ==========================================================

def filtro_average(img):
    """
    Filtro de média — único parâmetro: imagem uint8 grayscale.
    """

    if img.ndim != 2 or img.dtype != np.uint8:
        raise ValueError("Entrada inválida: esperada imagem uint8 2D.")

    # Janela 3x3 (valor fixo para cumprir o contrato de um só parâmetro)
    tamanho_janela = 3

    # Construir kernel de média explicitamente
    kernel = np.ones((tamanho_janela, tamanho_janela), dtype=np.float64)
    soma_pesos = tamanho_janela * tamanho_janela
    kernel = kernel / soma_pesos

    mapa_filtrado = convolucao_manual(img, kernel)
    return limitar_para_uint8(mapa_filtrado)

### Passa-baixo — Median (mediana)

Para cada janela, ordena valores e escolhe o elemento central.

In [ ]:
# ==========================================================
# FILTRO MEDIAN (PASSA-BAIXO)
# ==========================================================

def filtro_median(img):
    """
    Filtro da mediana — único parâmetro: imagem uint8 grayscale.
    """

    if img.ndim != 2 or img.dtype != np.uint8:
        raise ValueError("Entrada inválida: esperada imagem uint8 2D.")

    tamanho_janela = 3
    pad = tamanho_janela // 2
    altura, largura = img.shape
    imagem_padded = np.pad(img, pad, mode="edge")
    imagem_saida = np.zeros_like(img)

    for i in range(altura):
        for j in range(largura):
            janela = imagem_padded[i : i + tamanho_janela, j : j + tamanho_janela]
            lista_valores = janela.reshape(-1).tolist()
            lista_valores.sort()
            indice_mediana = len(lista_valores) // 2
            imagem_saida[i, j] = lista_valores[indice_mediana]

    return imagem_saida.astype(np.uint8)

### Passa-baixo — Gaussian

Kernel Gaussiano 2D construído explicitamente (sem funções prontas de suavização).

In [ ]:
# ==========================================================
# FILTRO GAUSSIAN (PASSA-BAIXO)
# ==========================================================

def filtro_gaussian(img):
    """
    Filtro Gaussiano — único parâmetro: imagem uint8 grayscale.
    """

    if img.ndim != 2 or img.dtype != np.uint8:
        raise ValueError("Entrada inválida: esperada imagem uint8 2D.")

    tamanho_janela = 3
    sigma = 1.0
    centro = tamanho_janela // 2
    kernel = np.zeros((tamanho_janela, tamanho_janela), dtype=np.float64)
    soma_kernel = 0.0

    for i in range(tamanho_janela):
        for j in range(tamanho_janela):
            x = float(j - centro)
            y = float(i - centro)
            peso = np.exp(-(x * x + y * y) / (2.0 * sigma * sigma))
            kernel[i, j] = peso
            soma_kernel += peso

    kernel = kernel / soma_kernel

    mapa_filtrado = convolucao_manual(img, kernel)
    return limitar_para_uint8(mapa_filtrado)

### Passa-alto — Sobel

Gradiente nas direções $G_x$ e $G_y$; magnitude $|G| = \sqrt{G_x^2 + G_y^2}$.

In [ ]:
# ==========================================================
# FILTRO SOBEL (PASSA-ALTO)
# ==========================================================

def filtro_sobel(img):
    """
    Operador de Sobel — único parâmetro: imagem uint8 grayscale.
    """

    if img.ndim != 2 or img.dtype != np.uint8:
        raise ValueError("Entrada inválida: esperada imagem uint8 2D.")

    kernel_gx = np.array(
        [
            [-1.0, 0.0, 1.0],
            [-2.0, 0.0, 2.0],
            [-1.0, 0.0, 1.0],
        ],
        dtype=np.float64,
    )

    kernel_gy = np.array(
        [
            [-1.0, -2.0, -1.0],
            [0.0, 0.0, 0.0],
            [1.0, 2.0, 1.0],
        ],
        dtype=np.float64,
    )

    gradiente_x = convolucao_manual(img, kernel_gx)
    gradiente_y = convolucao_manual(img, kernel_gy)

    altura, largura = img.shape
    magnitude = np.zeros((altura, largura), dtype=np.float64)

    for i in range(altura):
        for j in range(largura):
            gx = gradiente_x[i, j]
            gy = gradiente_y[i, j]
            magnitude[i, j] = np.sqrt(gx * gx + gy * gy)

    return limitar_para_uint8(magnitude)

### Passa-alto — Laplacian

Kernel discreto 3×3; utiliza valor absoluto da resposta e reescala para `uint8`.

In [ ]:
# ==========================================================
# FILTRO LAPLACIAN (PASSA-ALTO)
# ==========================================================

def filtro_laplacian(img):
    """
    Laplaciano discreto — único parâmetro: imagem uint8 grayscale.
    """

    if img.ndim != 2 or img.dtype != np.uint8:
        raise ValueError("Entrada inválida: esperada imagem uint8 2D.")

    kernel_laplacian = np.array(
        [
            [0.0, 1.0, 0.0],
            [1.0, -4.0, 1.0],
            [0.0, 1.0, 0.0],
        ],
        dtype=np.float64,
    )

    resposta = convolucao_manual(img, kernel_laplacian)

    altura, largura = img.shape
    resposta_absoluta = np.zeros((altura, largura), dtype=np.float64)

    for i in range(altura):
        for j in range(largura):
            resposta_absoluta[i, j] = abs(resposta[i, j])

    maximo = float(resposta_absoluta.max())
    if maximo <= 0:
        return img.copy()

    resposta_escalada = (resposta_absoluta / maximo) * 255.0
    return limitar_para_uint8(resposta_escalada)